 01_results_analysis — normalize + metrics + export

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = ROOT / "results"
PROCESSED_DIR = RESULTS_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_DATASETS = ["mmlu", "hellaswag", "gsm8k", "dolly"]


In [14]:
def discover_runs(results_dir):
    rows = []
    for model_dir in results_dir.iterdir():
        if not model_dir.is_dir() or model_dir.name == "processed":
            continue
        for run_dir in model_dir.iterdir():
            if run_dir.is_dir():
                rows.append({
                    "model": model_dir.name,
                    "run_id": run_dir.name,
                    "run_dir": run_dir
                })
    return pd.DataFrame(rows)

def load_all():
    runs = discover_runs(RESULTS_DIR)
    summaries, samples = [], []

    for _, r in runs.iterrows():
        for ds in EXPECTED_DATASETS:
            s = r.run_dir / f"{ds}.csv"
            sp = r.run_dir / f"{ds}_samples.csv"

            if s.exists():
                df = pd.read_csv(s)
                df["model_dir"] = r.model
                df["run_id"] = r.run_id
                df["dataset"] = ds
                summaries.append(df)

            if sp.exists():
                df = pd.read_csv(sp)
                df["model_dir"] = r.model
                df["run_id"] = r.run_id
                df["dataset"] = ds
                samples.append(df)

    return (
        pd.concat(summaries, ignore_index=True),
        pd.concat(samples, ignore_index=True)
    )

raw_summary, raw_samples = load_all()

print(raw_summary.shape, raw_samples.shape)


(15, 48) (12300, 26)


In [15]:
# normalize summary

MAP = {
    "Accuracy mean": "accuracy",
    "TTFT ms mean": "ttft",
    "End-to-end latency ms mean": "latency",
    "Tokens/sec mean": "tps",
    "CPU percent avg mean": "cpu",
    "Process RAM MB peak": "ram",
}

def normalize_summary(df):
    out = []

    for _, r in df.iterrows():
        row = {
            "model": r["model_dir"],
            "run_id": r["run_id"],
            "dataset": r["dataset"]
        }

        working = dict(r)

        if r["dataset"] == "dolly":
            working = {
                k.replace("Dolly ", ""): v for k, v in working.items()
            }

        for k, v in MAP.items():
            row[v] = working.get(k)

        out.append(row)

    return pd.DataFrame(out)

summary = normalize_summary(raw_summary)
for col in ["accuracy", "ttft", "latency", "tps", "cpu", "ram"]:
    if col in summary.columns:
        summary[col] = pd.to_numeric(summary[col], errors="coerce")

summary.head()


,model,run_id,dataset,accuracy,ttft,latency,tps,cpu,ram
0,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,mmlu,0.310,125.6957,316.5059,5.3356,65.7547,184.36
1,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,hellaswag,0.293,112.6663,304.7556,5.2812,66.1447,187.45
2,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,gsm8k,0.031,108.7212,770.5723,3.4579,74.4935,189.75
3,gemma-3-1b-it-Q8_0,2026-03-31_15-37-23,mmlu,0.310,125.7233,315.9056,5.3528,66.1230,184.25
4,gemma-3-1b-it-Q8_0,2026-03-31_15-37-23,hellaswag,0.293,112.1800,303.2874,5.3019,65.7051,177.03


In [16]:
# normalize samples

samples = raw_samples.rename(columns={
    "end_to_end_latency_ms": "latency",
})

samples["latency_s"] = samples["latency"] / 1000
samples["ttft_s"] = samples["ttft_ms"] / 1000

samples.head()


,model_name,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,latency,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,model_dir,run_id,dataset,latency_s,ttft_s
0,gemma-3-1b-it-Q8_0.gguf,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,B,False,B,1,106.0316,319.9848,4.6739,62.6333,94.3,3262.38,180.91,61.3000,62.6,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,mmlu,0.319985,0.106032
1,gemma-3-1b-it-Q8_0.gguf,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,107.0617,302.3457,5.1207,75.0000,98.7,3265.11,180.91,62.6000,62.6,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,mmlu,0.302346,0.107062
2,gemma-3-1b-it-Q8_0.gguf,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,106.6866,302.4155,5.1091,63.5667,97.5,3265.11,180.91,63.5000,64.2,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,mmlu,0.302416,0.106687
3,gemma-3-1b-it-Q8_0.gguf,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,C,False,C,1,110.4888,303.5208,5.1805,63.5000,97.5,3264.86,180.91,63.6667,64.2,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,mmlu,0.303521,0.110489
4,gemma-3-1b-it-Q8_0.gguf,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,C,False,C,1,112.5677,308.7586,5.0971,63.3333,98.7,3264.98,180.91,64.0667,64.8,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,mmlu,0.308759,0.112568


In [17]:
# derived metrics

summary["tps_per_ram"] = summary["tps"] / summary["ram"]
summary["accuracy_per_latency"] = summary["accuracy"] / summary["latency"]

summary.head()


,model,run_id,dataset,accuracy,ttft,latency,tps,cpu,ram,tps_per_ram,accuracy_per_latency
0,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,mmlu,0.310,125.6957,316.5059,5.3356,65.7547,184.36,0.028941,0.000979
1,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,hellaswag,0.293,112.6663,304.7556,5.2812,66.1447,187.45,0.028174,0.000961
2,gemma-3-1b-it-Q8_0,2026-03-31_10-52-54,gsm8k,0.031,108.7212,770.5723,3.4579,74.4935,189.75,0.018223,0.000040
3,gemma-3-1b-it-Q8_0,2026-03-31_15-37-23,mmlu,0.310,125.7233,315.9056,5.3528,66.1230,184.25,0.029052,0.000981
4,gemma-3-1b-it-Q8_0,2026-03-31_15-37-23,hellaswag,0.293,112.1800,303.2874,5.3019,65.7051,177.03,0.029949,0.000966


In [18]:
# model comparison

model_summary = (
    summary.groupby(["model", "dataset"])
    .mean(numeric_only=True)
    .reset_index()
)

model_summary


,model,dataset,accuracy,ttft,latency,tps,cpu,ram,tps_per_ram,accuracy_per_latency
0,Llama-3.2-1B-Instruct-Q8_0,dolly,NaN,2059.47890,8217.01890,6.01860,92.11320,193.520,0.031101,NaN
1,Llama-3.2-1B-Instruct-Q8_0,gsm8k,0.060,127.17640,298.79260,7.29070,68.61740,188.270,0.038725,0.000201
2,Llama-3.2-1B-Instruct-Q8_0,hellaswag,0.325,131.03240,248.72120,8.50400,66.98920,185.950,0.045733,0.001307
3,Llama-3.2-1B-Instruct-Q8_0,mmlu,0.411,150.37670,269.06230,8.53290,67.06880,182.910,0.046651,0.001528
4,Phi-4-mini-instruct-Q4_K_M,dolly,NaN,5582.97680,19030.23210,3.42190,95.91510,197.360,0.017338,NaN
5,Phi-4-mini-instruct-Q4_K_M,gsm8k,0.148,234.82930,813.91740,3.38390,74.34320,173.890,0.019460,0.000182
6,Phi-4-mini-instruct-Q4_K_M,hellaswag,0.509,250.97520,1106.59540,4.02480,72.03140,79.800,0.050436,0.000460
7,Phi-4-mini-instruct-Q4_K_M,mmlu,0.599,293.67920,1127.20510,3.55500,73.73750,145.000,0.024517,0.000531
8,gemma-3-1b-it-Q8_0,dolly,NaN,1283.05840,7133.57890,6.67640,92.86220,85.690,0.077913,NaN
9,gemma-3-1b-it-Q8_0,gsm8k,0.031,108.61335,769.58710,3.46225,74.38065,174.140,0.020045,0.000040


In [19]:
# scored tasks only

scored = summary[summary["dataset"].isin(["mmlu","hellaswag","gsm8k"])]

ranking = (
    scored.groupby("model")
    .agg(
        accuracy=("accuracy","mean"),
        latency=("latency","mean"),
        tps=("tps","mean"),
        ram=("ram","mean")
    )
    .reset_index()
    .sort_values("accuracy", ascending=False)
)

ranking


,model,accuracy,latency,tps,ram
1,Phi-4-mini-instruct-Q4_K_M,0.418667,1015.905967,3.654567,132.896667
0,Llama-3.2-1B-Instruct-Q8_0,0.265333,272.192033,8.109200,185.710000
2,gemma-3-1b-it-Q8_0,0.211333,463.271450,4.699333,180.228333


In [20]:
# save outputs

summary.to_csv(PROCESSED_DIR / "summary.csv", index=False)
samples.to_csv(PROCESSED_DIR / "samples.csv", index=False)
model_summary.to_csv(PROCESSED_DIR / "model_summary.csv", index=False)
ranking.to_csv(PROCESSED_DIR / "ranking.csv", index=False)

print("Saved to", PROCESSED_DIR)


Saved to /home/stephane/resource-efficiency-benchmark/results/processed


In [21]:
# %%